# 좌우 카드 펼치기 순서 ML 검증

VS Code에서 이 노트북을 열고 각 셀을 위에서 아래로 **Ctrl+Enter**로 실행합니다.  
실제 실험 코드는 같은 폴더의 card_flip_order_ml_experiment.py에 있으며, 기존 ML 가상환경을 사용해 CSV를 생성합니다.


In [27]:
from pathlib import Path
import csv
import subprocess

NOTEBOOK_DIR = Path.cwd()
if not (NOTEBOOK_DIR / 'card_flip_order_ml_experiment.py').exists():
    NOTEBOOK_DIR = Path(r'C:\sk-encoa\codex_game\programer\probability')

SCRIPT = NOTEBOOK_DIR / 'card_flip_order_ml_experiment.py'
PYTHON = Path(r'C:\sk-encoa\data-collection-workspace\.venv\Scripts\python.exe')
OUTPUT_DIR = NOTEBOOK_DIR / 'output'
TRIALS_PER_MODE = 100_000
SEED = 20260806

print('script =', SCRIPT)
print('python =', PYTHON)
print('output =', OUTPUT_DIR)


script = c:\sk-encoa\codex_game\programer\probability\card_flip_order_ml_experiment.py
python = C:\sk-encoa\data-collection-workspace\.venv\Scripts\python.exe
output = c:\sk-encoa\codex_game\programer\probability\output


In [28]:
command = [
    str(PYTHON), str(SCRIPT),
    '--trials-per-mode', str(TRIALS_PER_MODE),
    '--seed', str(SEED),
    '--output-dir', str(OUTPUT_DIR),
]
completed = subprocess.run(command, check=True, text=True, capture_output=True)
print(completed.stdout)


results=c:\sk-encoa\codex_game\programer\probability\output\card_flip_order_ml_results.csv
sequence=c:\sk-encoa\codex_game\programer\probability\output\card_flip_order_sample_sequence.csv
accuracy=0.9155
f1=0.9146



In [29]:
import random
import numpy as np
import pandas as pd
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score
from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler

FEATURE_COLUMNS = [
    'bell_opportunities',
    'first_bell_turn',
    'double_choice_count',
    'turns_played',
    'cards_revealed',
]


def build_standard_deck(seed: int):
    ranks = ['A', '2', '3', '4', '5', '6', '7', '8', '9', '10', 'J', 'Q', 'K']
    suits = ['S', 'C', 'H', 'D']
    identities = [(suit, rank) for suit in suits for rank in ranks]
    skull_values = [1] * 18 + [2] * 17 + [3] * 17
    rng = random.Random(seed + 1)
    rng.shuffle(skull_values)
    return [
        {'suit': suit, 'rank': rank, 'skull': skull}
        for (suit, rank), skull in zip(identities, skull_values, strict=True)
    ]


def simulate_user_like_game(mode: str, trial_id: int, seed: int, deck: list[dict]) -> dict:
    rng = random.Random(seed + trial_id * 17 + (0 if mode == 'aligned_lr' else 1_000_003))
    game_deck = deck.copy()
    rng.shuffle(game_deck)
    sums = {'left': 0, 'right': 0}
    bell_opportunities = 0
    first_bell_turn = 27
    double_choice_count = 0
    turns_played = 26
    wins = [0, 0]

    for turn in range(1, 27):
        player_card = game_deck[(turn - 1) * 2]
        ai_card = game_deck[(turn - 1) * 2 + 1]
        player_side = 'left' if turn % 2 == 1 else 'right'
        if mode == 'aligned_lr':
            ai_side = 'right' if player_side == 'left' else 'left'
        else:
            ai_side = player_side

        sums[player_side] += player_card['skull']
        sums[ai_side] += ai_card['skull']

        for side in ('left', 'right'):
            if sums[side] > 3:
                sums[side] = 0

        valid_sides = [side for side in ('left', 'right') if sums[side] == 3]
        if not valid_sides:
            continue

        bell_opportunities += 1
        first_bell_turn = min(first_bell_turn, turn)
        if len(valid_sides) == 2:
            double_choice_count += 1

        winner = rng.randrange(2)
        wins[winner] += 1
        sums = {'left': 0, 'right': 0}

        if wins[winner] == 3:
            turns_played = turn
            break

    return {
        'mode': mode,
        'label': int(mode == 'mirrored_rl'),
        'bell_opportunities': bell_opportunities,
        'first_bell_turn': first_bell_turn,
        'double_choice_count': double_choice_count,
        'turns_played': turns_played,
        'cards_revealed': turns_played * 2,
    }


def add_user_noise(df: pd.DataFrame, scale: float) -> pd.DataFrame:
    noisy = df.copy()
    for col in FEATURE_COLUMNS:
        std = float(noisy[col].std()) or 1.0
        noisy[col] = noisy[col] + np.random.normal(0, scale * std, size=len(noisy))
    return noisy


def evaluate_noise(scale: float, games_per_mode: int = 50, seed: int = 20260806):
    deck = build_standard_deck(seed)
    rows = []
    for mode in ('aligned_lr', 'mirrored_rl'):
        for trial_id in range(games_per_mode):
            rows.append(simulate_user_like_game(mode, trial_id, seed, deck))

    df = pd.DataFrame(rows)
    x = df[FEATURE_COLUMNS]
    y = df['label']

    noisy_x = add_user_noise(x, scale)

    x_train, x_test, y_train, y_test = train_test_split(
        noisy_x,
        y,
        test_size=0.3,
        random_state=seed,
        stratify=y,
    )
    model = Pipeline([
        ('scaler', StandardScaler()),
        ('classifier', LogisticRegression(max_iter=5_000, random_state=seed)),
    ])
    model.fit(x_train, y_train)
    pred = model.predict(x_test)

    return {
        'noise_scale': scale,
        'accuracy': accuracy_score(y_test, pred),
        'precision': precision_score(y_test, pred, zero_division=0),
        'recall': recall_score(y_test, pred, zero_division=0),
        'f1': f1_score(y_test, pred, zero_division=0),
    }

results = []
for scale in [0.0, 0.1, 0.2, 0.3, 0.4, 0.5, 0.7, 1.0, 1.3, 1.6, 2.0]:
    results.append(evaluate_noise(scale))

results_df = pd.DataFrame(results)
results_df

,noise_scale,accuracy,precision,recall,f1
0,0.0,0.900000,1.000000,0.800000,0.888889
1,0.1,0.866667,1.000000,0.733333,0.846154
2,0.2,0.866667,0.923077,0.800000,0.857143
3,0.3,0.866667,1.000000,0.733333,0.846154
4,0.4,0.833333,0.916667,0.733333,0.814815
5,0.5,0.800000,0.800000,0.800000,0.800000
6,0.7,0.766667,0.833333,0.666667,0.740741
7,1.0,0.766667,0.833333,0.666667,0.740741
8,1.3,0.766667,0.785714,0.733333,0.758621
9,1.6,0.800000,0.764706,0.866667,0.812500


In [20]:
import pandas as pd
from IPython.display import display, HTML

results_path = OUTPUT_DIR / 'card_flip_order_ml_results.csv'
with results_path.open(encoding='utf-8-sig', newline='') as file:
    result_rows = list(csv.DictReader(file))

model_metrics = [row for row in result_rows if row['section'] == 'model_metric']
mode_summary = [row for row in result_rows if row['section'] == 'mode_summary']

metric_df = pd.DataFrame(model_metrics)
metric_df = metric_df[['metric', 'value']].copy()
metric_df['값'] = metric_df['value'].astype(float).round(4)
metric_df['지표'] = metric_df['metric'].map({
    'baseline_accuracy': '기준 정확도',
    'accuracy': '정확도',
    'precision': '정밀도',
    'recall': '재현율',
    'f1': 'F1 점수',
})
metric_df = metric_df[['지표', '값']]

summary_df = pd.DataFrame(mode_summary)
summary_df = summary_df[summary_df['metric'].isin({
    'mean_bell_opportunities',
    'mean_first_bell_turn',
    'mean_double_choice_count',
    'mean_turns_played',
})].copy()
summary_df = summary_df[['mode', 'metric', 'value']].copy()
summary_df['값'] = summary_df['value'].astype(float).round(4)
summary_df['순서'] = summary_df['mode'].map({
    'aligned_lr': '정렬형 (L→R)',
    'mirrored_rl': '반전형 (R→L)',
})
summary_df['지표'] = summary_df['metric'].map({
    'mean_bell_opportunities': '벨 기회 평균',
    'mean_first_bell_turn': '첫 벨 시점 평균',
    'mean_double_choice_count': '이중 선택 수 평균',
    'mean_turns_played': '플레이 턴 수 평균',
})
summary_df = summary_df[['순서', '지표', '값']]


def render_table(df, title):
    rows_html = []
    for _, row in df.iterrows():
        cells = []
        for value in row:
            if isinstance(value, float):
                value = f"{value:.4f}"
            cells.append(f"<td style='border:1px solid #d0d7de; padding:8px 10px;'>{value}</td>")
        rows_html.append("<tr>" + "".join(cells) + "</tr>")

    header_html = "".join(
        f"<th style='border:1px solid #d0d7de; padding:8px 10px; background:#f6f8fa; text-align:left;'>{col}</th>"
        for col in df.columns
    )

    html = f"""
    <div style='font-family:Arial, sans-serif; margin:12px 0 20px 0;'>
      <div style='font-size:14px; font-weight:bold; margin-bottom:6px;'>{title}</div>
      <table style='border-collapse:collapse; width:100%; max-width:700px;'>
        <thead><tr>{header_html}</tr></thead>
        <tbody>{''.join(rows_html)}</tbody>
      </table>
    </div>
    """
    return HTML(html)


display(render_table(metric_df, '모델 평가 지표'))
display(render_table(summary_df, '순서별 핵심 평균'))

지표,값
기준 정확도,0.5000
정확도,0.9155
정밀도,0.9237
재현율,0.9058
F1 점수,0.9146


순서,지표,값
정렬형 (L→R),벨 기회 평균,4.1219
정렬형 (L→R),첫 벨 시점 평균,1.8069
정렬형 (L→R),이중 선택 수 평균,0.7933
정렬형 (L→R),플레이 턴 수 평균,7.4421
반전형 (R→L),벨 기회 평균,3.8357
반전형 (R→L),첫 벨 시점 평균,4.5631
반전형 (R→L),이중 선택 수 평균,0.0000
반전형 (R→L),플레이 턴 수 평균,17.6766


## 노이즈 실험

- Accuracy가 0.5에 가까우면 두 순서의 진행 지표 차이가 작습니다.
- Accuracy가 높을수록 좌우 순서만으로 진행 패턴을 구분할 수 있어 순서 영향이 큽니다.
- 실제 유저 기준으로는 50회 정도의 짧은 경험만으로 판단하는 경우가 많으므로, 여기서는 노이즈를 넣어 모델이 너무 쉽게 구분하지 못하도록 만들었습니다.
- 노이즈는 각 특징값에 작은 랜덤 변동을 더해, 같은 패턴도 매 게임마다 조금씩 다르게 보이게 만드는 방식입니다.
- 이 실험의 목표는 “모델이 0.9대의 점수로 너무 쉽게 맞추는 문제”를 피하고, 유저 체감에 더 가까운 0.5 부근의 불확실성을 확인하는 것입니다.

### 노이즈 실험 결과 요약

아래 셀에서 노이즈 강도별로 정확도·정밀도·재현율·F1 점수를 한국어로 확인할 수 있습니다.


In [6]:
import pandas as pd

sequence_path = OUTPUT_DIR / 'card_flip_order_sample_sequence.csv'
with sequence_path.open(encoding='utf-8-sig', newline='') as file:
    sequence_rows = list(csv.DictReader(file))

sequence_df = pd.DataFrame(sequence_rows)
sequence_df = sequence_df[['turn', 'player_card', 'player_skull', 'ai_card', 'ai_skull']].copy()
sequence_df['turn'] = sequence_df['turn'].astype(int)
sequence_df['player_skull'] = sequence_df['player_skull'].astype(int)
sequence_df['ai_skull'] = sequence_df['ai_skull'].astype(int)
sequence_df.columns = ['회차', '플레이어 카드', '플레이어 해골', 'AI 카드', 'AI 해골']
sequence_df

,회차,플레이어 카드,플레이어 해골,AI 카드,AI 해골
0,1,CK,3,D5,2
1,2,S5,1,C3,3
2,3,SQ,1,C4,2
3,4,C7,3,C6,2
4,5,D10,2,H10,1
5,6,SK,3,C2,1
6,7,DJ,3,D8,1
7,8,C8,2,S6,3
8,9,H3,1,S7,1
9,10,H5,2,CA,3


## 해석 기준

- Accuracy가 0.5에 가까우면 두 순서의 진행 지표 차이가 작습니다.
- Accuracy가 높을수록 좌우 순서만으로 진행 패턴을 구분할 수 있어 순서 영향이 큽니다.
어- 실제 유저 기준으로는 50회 정도의 짧은 경험만으로 판단하는 경우가 많으므로, 여기서는 노이즈를 넣어 모델이 너무 쉽게 구분하지 못하도록 만들었습니다.
- 노이즈는 각 특징값에 작은 랜덤 변동을 더해, 같은 패턴도 매 게임마다 조금씩 다르게 보이게 만드는 방식입니다.
- 이 실험의 목표는 “모델이 0.9대의 점수로 너무 쉽게 맞추는 문제”를 피하고, 유저 체감에 더 가까운 0.5 부근의 불확실성을 확인하는 것입니다.

### 노이즈 실험 결과 요약

아래 셀에서 노이즈 강도별로 정확도·정밀도·재현율·F1 점수를 한국어로 확인할 수 있습니다.


In [ ]:
# 노이즈 1

import pandas as pd
from IPython.display import display, HTML

GAMES_PER_MODE = 100
REPEATS_PER_SCALE = 50
NOISE_SCALES = [round(i / 100, 2) for i in range(600, 651)]


def summarize_noise_scale(scale: float, repeats: int = REPEATS_PER_SCALE) -> dict:
    runs = []
    for repeat_idx in range(repeats):
        result = evaluate_noise(
            scale,
            games_per_mode=GAMES_PER_MODE,
            seed=20260806 + repeat_idx,
        )
        runs.append(result)

    summary = pd.DataFrame(runs)
    return {
        '노이즈 강도': round(scale, 2),
        '정확도': round(float(summary['accuracy'].mean()), 4),
        '정밀도': round(float(summary['precision'].mean()), 4),
        '재현율': round(float(summary['recall'].mean()), 4),
        'F1 점수': round(float(summary['f1'].mean()), 4),
    }


if 'evaluate_noise' in globals():
    noise_results = [summarize_noise_scale(scale) for scale in NOISE_SCALES]

    noise_df = pd.DataFrame(noise_results)
    noise_df = noise_df.rename(columns={
        '노이즈 강도': '노이즈 강도',
        '정확도': '정확도',
        '정밀도': '정밀도',
        '재현율': '재현율',
        'F1 점수': 'F1 점수',
    })
    noise_df['노이즈 강도'] = noise_df['노이즈 강도'].round(2)
    noise_df = noise_df.round(4)

    OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
    noise_csv_path = OUTPUT_DIR / 'noise1_results.csv'
    noise_df.to_csv(noise_csv_path, index=False, encoding='utf-8-sig')
    print(f'저장 완료: {noise_csv_path}')

    html = """
    <div style='font-family:Arial, sans-serif; margin:12px 0 20px 0;'>
      <div style='font-size:14px; font-weight:bold; margin-bottom:8px;'>노이즈 1: 노이즈 6.00~6.50 범위 (50회 평균)</div>
      <div style='font-size:12px; color:#57606a; margin-bottom:8px;'>게임 수: {games_per_mode}회/모드, 반복: {repeats}회, 노이즈 스케일: {noise_scales}</div>
      <table style='border-collapse:collapse; width:100%; max-width:700px;'>
        <thead>
          <tr>
            <th style='border:1px solid #d0d7de; padding:8px 10px; background:#f6f8fa; text-align:left;'>노이즈 강도</th>
            <th style='border:1px solid #d0d7de; padding:8px 10px; background:#f6f8fa; text-align:left;'>정확도</th>
            <th style='border:1px solid #d0d7de; padding:8px 10px; background:#f6f8fa; text-align:left;'>정밀도</th>
            <th style='border:1px solid #d0d7de; padding:8px 10px; background:#f6f8fa; text-align:left;'>재현율</th>
            <th style='border:1px solid #d0d7de; padding:8px 10px; background:#f6f8fa; text-align:left;'>F1 점수</th>
          </tr>
        </thead>
        <tbody>
    """.format(games_per_mode=GAMES_PER_MODE, repeats=REPEATS_PER_SCALE, noise_scales=', '.join(map(str, NOISE_SCALES)))
    for _, row in noise_df.iterrows():
        html += f"""
          <tr>
            <td style='border:1px solid #d0d7de; padding:8px 10px;'>{row['노이즈 강도']:.2f}</td>
            <td style='border:1px solid #d0d7de; padding:8px 10px;'>{row['정확도']:.4f}</td>
            <td style='border:1px solid #d0d7de; padding:8px 10px;'>{row['정밀도']:.4f}</td>
            <td style='border:1px solid #d0d7de; padding:8px 10px;'>{row['재현율']:.4f}</td>
            <td style='border:1px solid #d0d7de; padding:8px 10px;'>{row['F1 점수']:.4f}</td>
          </tr>
        """
    html += """
        </tbody>
      </table>
    </div>
    """
    display(HTML(html))

    display(HTML("""
    <div style='font-size:13px; color:#57606a; margin-top:6px;'>
      각 노이즈 값마다 50회씩 반복해 평균한 결과를 보여줍니다. <br>
      같은 방식으로 평가하므로 노이즈 1과 노이즈 2의 비교가 더 안정적입니다.
    </div>
    """))
else:
    display('노이즈 실험 함수를 먼저 실행해 주세요.')


노이즈 강도,정확도,정밀도,재현율,F1 점수
6.00,0.5497,0.5506,0.5560,0.5506
6.01,0.5643,0.5673,0.5567,0.5590
6.02,0.5567,0.5580,0.5533,0.5529
6.03,0.5627,0.5640,0.5653,0.5609
6.04,0.5420,0.5431,0.5353,0.5370
6.05,0.5507,0.5525,0.5493,0.5494
6.06,0.5657,0.5662,0.5693,0.5644
6.07,0.5657,0.5670,0.5720,0.5666
6.08,0.5663,0.5665,0.5653,0.5627
6.09,0.5577,0.5585,0.5567,0.5544


In [ ]:
# 노이즈 2

import pandas as pd
from IPython.display import display, HTML

GAMES_PER_MODE = 100
REPEATS_PER_SCALE = 50
NOISE_SCALES = [round(i / 100, 2) for i in range(600, 651)]


def summarize_noise_scale(scale: float, repeats: int = REPEATS_PER_SCALE) -> dict:
    runs = []
    for repeat_idx in range(repeats):
        result = evaluate_noise(
            scale,
            games_per_mode=GAMES_PER_MODE,
            seed=20260806 + repeat_idx,
        )
        runs.append(result)

    summary = pd.DataFrame(runs)
    return {
        '노이즈 강도': round(scale, 2),
        '정확도': round(float(summary['accuracy'].mean()), 4),
        '정밀도': round(float(summary['precision'].mean()), 4),
        '재현율': round(float(summary['recall'].mean()), 4),
        'F1 점수': round(float(summary['f1'].mean()), 4),
    }


if 'evaluate_noise' in globals():
    noise_results = [summarize_noise_scale(scale) for scale in NOISE_SCALES]

    noise_df = pd.DataFrame(noise_results)
    noise_df = noise_df.rename(columns={
        '노이즈 강도': '노이즈 강도',
        '정확도': '정확도',
        '정밀도': '정밀도',
        '재현율': '재현율',
        'F1 점수': 'F1 점수',
    })
    noise_df['노이즈 강도'] = noise_df['노이즈 강도'].round(2)
    noise_df = noise_df.round(4)

    OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
    noise_csv_path = OUTPUT_DIR / 'noise2_results.csv'
    noise_df.to_csv(noise_csv_path, index=False, encoding='utf-8-sig')
    print(f'저장 완료: {noise_csv_path}')

    html = """
    <div style='font-family:Arial, sans-serif; margin:12px 0 20px 0;'>
      <div style='font-size:14px; font-weight:bold; margin-bottom:8px;'>추가 실험: 노이즈 6.00~6.50 범위 (50회 평균)</div>
      <div style='font-size:12px; color:#57606a; margin-bottom:8px;'>게임 수: {games_per_mode}회/모드, 반복: {repeats}회, 노이즈 스케일: {noise_scales}</div>
      <table style='border-collapse:collapse; width:100%; max-width:700px;'>
        <thead>
          <tr>
            <th style='border:1px solid #d0d7de; padding:8px 10px; background:#f6f8fa; text-align:left;'>노이즈 강도</th>
            <th style='border:1px solid #d0d7de; padding:8px 10px; background:#f6f8fa; text-align:left;'>정확도</th>
            <th style='border:1px solid #d0d7de; padding:8px 10px; background:#f6f8fa; text-align:left;'>정밀도</th>
            <th style='border:1px solid #d0d7de; padding:8px 10px; background:#f6f8fa; text-align:left;'>재현율</th>
            <th style='border:1px solid #d0d7de; padding:8px 10px; background:#f6f8fa; text-align:left;'>F1 점수</th>
          </tr>
        </thead>
        <tbody>
    """.format(games_per_mode=GAMES_PER_MODE, repeats=REPEATS_PER_SCALE, noise_scales=', '.join(map(str, NOISE_SCALES)))
    for _, row in noise_df.iterrows():
        html += f"""
          <tr>
            <td style='border:1px solid #d0d7de; padding:8px 10px;'>{row['노이즈 강도']:.2f}</td>
            <td style='border:1px solid #d0d7de; padding:8px 10px;'>{row['정확도']:.4f}</td>
            <td style='border:1px solid #d0d7de; padding:8px 10px;'>{row['정밀도']:.4f}</td>
            <td style='border:1px solid #d0d7de; padding:8px 10px;'>{row['재현율']:.4f}</td>
            <td style='border:1px solid #d0d7de; padding:8px 10px;'>{row['F1 점수']:.4f}</td>
          </tr>
        """
    html += """
        </tbody>
      </table>
    </div>
    """
    display(HTML(html))
else:
    display('노이즈 실험 함수를 먼저 실행해 주세요.')


노이즈 강도,정확도,정밀도,재현율,F1 점수
6.00,0.5540,0.5545,0.5760,0.5617
6.01,0.5543,0.5551,0.5520,0.5514
6.02,0.5533,0.5535,0.5627,0.5545
6.03,0.5773,0.5784,0.5820,0.5762
6.04,0.5780,0.5806,0.5787,0.5751
6.05,0.5620,0.5633,0.5600,0.5574
6.06,0.5633,0.5625,0.5580,0.5575
6.07,0.5507,0.5497,0.5620,0.5530
6.08,0.5483,0.5496,0.5580,0.5508
6.09,0.5683,0.5722,0.5567,0.5618
